# Radar Basics: step-by-step coding notebook

这个 notebook 会按小步推进的方式学习 `radar_basics` codebase。每一步我们只写少量代码，先运行、观察，再解释它在 radar simulation mental model 里代表什么。

## Step 1: YAML 是实验说明书

先不要急着进入 `RadarSystem` 或信号处理。这个项目的入口是一份 YAML config：它描述一次雷达仿真实验需要哪些东西。

这一步我们只做三件事：

1. 找到 `example_configs/architecture.yaml`。
2. 把 YAML 读成 Python dict。
3. 看清楚它有哪些 top-level sections。

Mental model: YAML 不是仿真结果，也不是雷达对象本身；它是一次 experiment 的 declarative specification。

In [11]:
from pathlib import Path
from pprint import pprint

import yaml


def find_project_root(start: Path | None = None) -> Path:
    """Find the repository root from either the repo folder or the learning folder."""
    start = start or Path.cwd()
    for candidate in (start, *start.parents):
        config_path = candidate / "example_configs" / "architecture.yaml"
        if (candidate / "pyproject.toml").exists() and config_path.exists():
            return candidate
    raise FileNotFoundError("Could not find the radar_basics project root")


PROJECT_ROOT = find_project_root()
CONFIG_PATH = PROJECT_ROOT / "example_configs" / "architecture.yaml"

with CONFIG_PATH.open("r", encoding="utf-8") as handle:
    raw_config = yaml.safe_load(handle)

print(f"Project root: {PROJECT_ROOT}")
print(f"Config path:   {CONFIG_PATH}")

print("\nTop-level sections:")
for section_name in raw_config:
    print(f"  - {section_name}")

print("\nYAML loaded as a Python dict:")
pprint(raw_config, sort_dicts=False)

Project root: c:\Users\yanzhang\Desktop\Research_Projects\radar_basics
Config path:   c:\Users\yanzhang\Desktop\Research_Projects\radar_basics\example_configs\architecture.yaml

Top-level sections:
  - radar
  - array
  - waveform
  - scan
  - scene
  - processing
  - run

YAML loaded as a Python dict:
{'radar': {'carrier_frequency_hz': 10000000000.0,
           'peak_tx_power_w': 50000.0,
           'noise_figure_db': 0.0,
           'system_loss_db': 0.0,
           'temperature_k': 0.0},
 'array': {'num_y': 4, 'num_x': 4},
 'waveform': {'bandwidth_hz': 5000000.0,
              'pulse_width_s': 2e-06,
              'prf_hz': 10000.0,
              'num_pulses': 16,
              'sample_rate_hz': 10000000.0},
 'scan': {'azimuths_deg': [-10.0, 0.0, 10.0],
          'elevations_deg': [-5.0, 0.0, 5.0],
          'mode': 'search'},
 'scene': {'targets': [{'name': 'target-a',
                        'range_m': 1500.0,
                        'az_deg': 0.0,
                        'el_deg'

### 观察这个输出

请先重点看 top-level sections，而不是陷入每个数字的细节。

- `radar`：雷达本机参数，比如 carrier frequency、peak transmit power、noise/loss。
- `array`：阵列几何，当前是二维 rectangular planar array。
- `waveform`：LFM pulse waveform 和 CPI 相关参数。
- `scan`：雷达要看哪些 azimuth/elevation 方向。
- `scene`：truth world，里面有哪些目标。
- `processing`：处理链路的角度网格、detector、tracker 配置。
- `run`：这次实验跑几轮 scan、随机种子、是否保存 raw IQ。

暂停点：运行上面的 cell 后，确认你能回答这个问题：如果我只改 YAML 里的 `scene.targets[0].range_m`，我是在改雷达硬件、改目标真值，还是改信号处理算法？

## Step 2: Config -> Python Objects

Step 1 里，`yaml.safe_load()` 得到的是普通 nested dictionary。Python 可以处理它，但这个 dict 还没有被这个 codebase 理解成一个正式的 radar experiment。

现在我们用 `load_config()` 做下一层转换：

```text
YAML file -> nested dict -> ExperimentConfig dataclass
```

`ExperimentConfig` 的价值是：它把 loose 的 dict 变成有结构、有字段名、经过基本校验的 Python object。

In [12]:
from dataclasses import fields, is_dataclass

from radar_basics import load_config


config = load_config(CONFIG_PATH)

print(f"raw_config type: {type(raw_config).__name__}")
print(f"config type:     {type(config).__name__}")
print(f"is dataclass:    {is_dataclass(config)}")

print("\nTop-level config objects:")
for field in fields(config):
    value = getattr(config, field.name)
    print(f"  {field.name:<10} -> {type(value).__name__}")

print("\nA few concrete fields:")
print(f"carrier_frequency_hz = {config.radar.carrier_frequency_hz:,.1f}")
print(f"array size           = {config.array.num_y} x {config.array.num_x}")
print(f"num_pulses           = {config.waveform.num_pulses}")
print(f"scan azimuths        = {config.scan.azimuths_deg}")
print(f"scan elevations      = {config.scan.elevations_deg}")
print(f"target count         = {len(config.scene.targets)}")
print(f"run scan cycles      = {config.run.num_scan_cycles}")

raw_config type: dict
config type:     ExperimentConfig
is dataclass:    True

Top-level config objects:
  radar      -> RadarConfig
  array      -> ArrayConfig
  waveform   -> WaveformConfig
  scan       -> ScanConfig
  scene      -> SceneConfig
  processing -> ProcessingConfig
  run        -> RunConfig

A few concrete fields:
carrier_frequency_hz = 10,000,000,000.0
array size           = 4 x 4
num_pulses           = 16
scan azimuths        = (-10.0, 0.0, 10.0)
scan elevations      = (-5.0, 0.0, 5.0)
target count         = 1
run scan cycles      = 2


### 观察这个输出

这里有一个重要分层：

- `raw_config` 是普通 `dict`，来自 YAML parser。
- `config` 是 `ExperimentConfig`，来自 `radar_basics.config.load_config()`。
- `config.radar`、`config.array`、`config.waveform` 等字段也各自是 dataclass object。

这一步还没有创建真正会参与仿真的 runtime objects。比如现在的 `config.radar` 只是雷达配置；下一步 Step 3A 里，我们会用它 build 出真正的 `RadarSystem`。

暂停点：请确认你能区分这三层：YAML text、Python dict、`ExperimentConfig` dataclass。

### My understanding after Step 2

到目前为止，可以把这三种形式理解成同一份 simulation specification 的不同表示：

```text
YAML text -> raw_config dictionary -> ExperimentConfig dataclass
```

它们表达的信息本质上是一样的：都是这次 radar simulation 需要的参数。区别主要在数据组织方式：

- YAML 方便人类阅读和编辑。
- `raw_config` 是 Python 可以直接处理的 nested `dict` / `list`。
- `ExperimentConfig` 是更结构化的 dataclass object，把 `radar`、`array`、`waveform`、`scan`、`scene`、`processing`、`run` 分别放进对应的 config dataclass。

这个 dataclass abstraction 对小项目来说看起来有点重，但它提供了字段结构、基本校验和更清楚的代码接口。

真正重要的边界是下一步：

```text
ExperimentConfig -> RadarSystem / Scene / Scheduler
```

从这里开始，就不只是同一份参数换一种组织方式了，而是会 build 出有行为的 runtime objects。比如 `RadarSystem` 可以计算 wavelength/range resolution，`Scene` 可以在某个时间产生 target snapshot，`Scheduler` 可以生成 dwell tasks。

## Step 3A: Radar Module

现在开始从 config specification 进入 runtime objects。第一站是 `radar_basics.radar`。

这一步我们只学习 radar 这一层：

- `config.radar`、`config.array`、`config.waveform` 是配置参数。
- `build_radar_system(config)` 会把这些参数 build 成一个 `RadarSystem`。
- `RadarSystem` 里面包含真正参与仿真的 `RectangularArray` 和 `LfmPulseWaveform`。

Mental model: `RadarSystem` 代表“这台雷达是什么样的”。它不包含目标，也不包含扫描任务；它只描述硬件、阵列、波形和由这些参数推导出来的雷达能力。

In [13]:
from radar_basics import build_radar_system


radar = build_radar_system(config)
array = radar.array
waveform = radar.waveform

print("Config objects -> runtime object")
print(f"config.radar type:    {type(config.radar).__name__}")
print(f"config.array type:    {type(config.array).__name__}")
print(f"config.waveform type: {type(config.waveform).__name__}")
print(f"radar type:           {type(radar).__name__}")

print("\nRadarSystem: derived quantities")
print(f"carrier_frequency_hz       = {radar.carrier_frequency_hz:,.1f} Hz")
print(f"wavelength_m               = {radar.wavelength_m:.6f} m")
print(f"max_unambiguous_range_m    = {radar.max_unambiguous_range_m:,.2f} m")
print(f"range_resolution_m         = {radar.range_resolution_m:.2f} m")
print(f"velocity_resolution_mps    = {radar.velocity_resolution_mps:.2f} m/s")
print(f"max_unambiguous_velocity_mps = {radar.max_unambiguous_velocity_mps:.2f} m/s")
print(f"noise_power_w              = {radar.noise_power_w:.3e} W")

print("\nRectangularArray: element geometry")
print(f"array type        = {type(array).__name__}")
print(f"num_y x num_x     = {array.num_y} x {array.num_x}")
print(f"num_elements      = {array.num_elements}")
print(f"spacing_y_m       = {array.spacing_y_m:.6f} m")
print(f"spacing_x_m       = {array.spacing_x_m:.6f} m")
print(f"positions_m shape = {array.positions_m.shape}")
print(f"corner element 0  = {array.positions_m[0, 0]}")
print(f"corner element -1 = {array.positions_m[-1, -1]}")

print("\nLfmPulseWaveform: pulse and CPI settings")
print(f"waveform type          = {type(waveform).__name__}")
print(f"bandwidth_hz           = {waveform.bandwidth_hz:,.1f} Hz")
print(f"pulse_width_s          = {waveform.pulse_width_s:.3e} s")
print(f"prf_hz                 = {waveform.prf_hz:,.1f} Hz")
print(f"pri_s                  = {waveform.pri_s:.3e} s")
print(f"num_pulses             = {waveform.num_pulses}")
print(f"sample_rate_hz         = {waveform.sample_rate_hz:,.1f} Hz")
print(f"num_fast_time_samples  = {waveform.num_fast_time_samples}")
print(f"num_pulse_samples      = {waveform.num_pulse_samples}")

Config objects -> runtime object
config.radar type:    RadarConfig
config.array type:    ArrayConfig
config.waveform type: WaveformConfig
radar type:           RadarSystem

RadarSystem: derived quantities
carrier_frequency_hz       = 10,000,000,000.0 Hz
wavelength_m               = 0.029979 m
max_unambiguous_range_m    = 14,989.62 m
range_resolution_m         = 29.98 m
velocity_resolution_mps    = 9.37 m/s
max_unambiguous_velocity_mps = 74.95 m/s
noise_power_w              = 0.000e+00 W

RectangularArray: element geometry
array type        = RectangularArray
num_y x num_x     = 4 x 4
num_elements      = 16
spacing_y_m       = 0.014990 m
spacing_x_m       = 0.014990 m
positions_m shape = (4, 4, 3)
corner element 0  = [ 0.         -0.02248443 -0.02248443]
corner element -1 = [0.         0.02248443 0.02248443]

LfmPulseWaveform: pulse and CPI settings
waveform type          = LfmPulseWaveform
bandwidth_hz           = 5,000,000.0 Hz
pulse_width_s          = 2.000e-06 s
prf_hz              

### 观察这个输出

这里开始出现真正的 runtime behavior：

- `config.radar` 是 `RadarConfig`，只是参数容器。
- `radar` 是 `RadarSystem`，它可以根据参数计算 `wavelength_m`、`range_resolution_m`、`velocity_resolution_mps` 等派生量。
- `radar.array` 是 `RectangularArray`，它知道阵元数量、间距和每个阵元的三维坐标。
- `radar.waveform` 是 `LfmPulseWaveform`，它知道 PRI、fast-time samples、pulse samples 等波形行为。

请先不要背公式，先抓住这个边界：YAML/config 只保存参数；`RadarSystem` 开始把参数变成可以计算、可以参与仿真的对象。

暂停点：请确认你能解释 `config.waveform.num_pulses` 和 `radar.waveform.num_pulses` 的关系，以及为什么 `radar.wavelength_m` 不需要直接写在 YAML 里。

### My understanding after Step 3A

就是说 step 3A 这个 radar object 表示的就是硬件上的雷达系统：包括 antenna array (存储的信息有： 每一个 element 的位置， 每一个 element 的 steering vector), transmit gain 的信息；包括 waveform （存储的信息有：lfm 的参数， pri, fast-time, slow-time 这些信息， 还有就是一个 chirp 的 sample values 这些。；包括其他一些 radar performance specs 的参数（carrier freq, unambiguous range, ... 这些）。

## Step 3B: Scene Module

现在学习 `radar_basics.scenario`。如果 Step 3A 的 `radar` 描述“这台雷达是什么样的”，那么 Step 3B 的 `scene` 描述“外部世界里有什么目标”。

这一步我们只学习 truth world：

- `config.scene` 是场景配置。
- `build_scene(config)` 会创建 runtime `Scene`。
- `Scene` 里面包含一个或多个 `PointTarget`。
- `Scene.snapshots_at(time_s)` 会在指定时间生成 `TargetSnapshot`。

Mental model: `Scene` 不是 measurement，也不是 detection。它是真值层，告诉仿真器目标真实在哪里、怎么运动、RCS 是多少。

In [14]:
from radar_basics import build_scene


scene = build_scene(config)

print("Config object -> runtime object")
print(f"config.scene type: {type(config.scene).__name__}")
print(f"scene type:        {type(scene).__name__}")
print(f"target count:      {len(scene.targets)}")

print("\nPointTarget objects stored inside Scene:")
for target in scene.targets:
    print(f"target type:   {type(target).__name__}")
    print(f"name:          {target.name}")
    print(f"position_m:    {target.position_m}")
    print(f"velocity_mps:  {target.velocity_mps}")
    print(f"rcs_m2:        {target.rcs_m2}")

snapshot_t0 = scene.snapshots_at(0.0)[0]
snapshot_t1 = scene.snapshots_at(1.0)[0]

print("\nTargetSnapshot at t = 0.0 s:")
print(f"snapshot type:        {type(snapshot_t0).__name__}")
print(f"name:                 {snapshot_t0.name}")
print(f"position_m:           {snapshot_t0.position_m}")
print(f"velocity_mps:         {snapshot_t0.velocity_mps}")
print(f"range_m:              {snapshot_t0.range_m:.2f}")
print(f"az_deg:               {snapshot_t0.az_deg:.2f}")
print(f"el_deg:               {snapshot_t0.el_deg:.2f}")
print(f"radial_velocity_mps:  {snapshot_t0.radial_velocity_mps:.2f}")
print(f"rcs_m2:               {snapshot_t0.rcs_m2}")

print("\nCompare t = 0.0 s and t = 1.0 s:")
print(f"position at t0: {snapshot_t0.position_m}")
print(f"position at t1: {snapshot_t1.position_m}")
print(f"range at t0:    {snapshot_t0.range_m:.2f} m")
print(f"range at t1:    {snapshot_t1.range_m:.2f} m")

Config object -> runtime object
config.scene type: SceneConfig
scene type:        Scene
target count:      1

PointTarget objects stored inside Scene:
target type:   PointTarget
name:          target-a
position_m:    (1500.0, 0.0, 0.0)
velocity_mps:  (0.0, 0.0, 0.0)
rcs_m2:        10.0

TargetSnapshot at t = 0.0 s:
snapshot type:        TargetSnapshot
name:                 target-a
position_m:           [1500.    0.    0.]
velocity_mps:         [0. 0. 0.]
range_m:              1500.00
az_deg:               0.00
el_deg:               0.00
radial_velocity_mps:  0.00
rcs_m2:               10.0

Compare t = 0.0 s and t = 1.0 s:
position at t0: [1500.    0.    0.]
position at t1: [1500.    0.    0.]
range at t0:    1500.00 m
range at t1:    1500.00 m


### 观察这个输出

这里的关键边界是：

- `PointTarget` 保存目标的基础真值：名字、三维位置、三维速度、RCS。
- `TargetSnapshot` 是某个具体时间点的目标状态，会额外计算出 `range_m`、`az_deg`、`el_deg`、`radial_velocity_mps`。
- `radial_velocity_mps` 是速度在 radar line-of-sight 方向上的分量，不一定等于完整三维速度的大小。

当前 example config 里的目标径向速度是 `0.0 m/s`，所以 `t = 0.0 s` 和 `t = 1.0 s` 的 position/range 应该相同。

暂停点：请确认你能解释 `PointTarget` 和 `TargetSnapshot` 的区别，以及为什么 `Scene` 是 truth world 而不是 radar measurement。

## Step 3C: Scheduler Module

现在学习 `radar_basics.scheduler`。如果 `radar` 是硬件和波形，`scene` 是真实世界目标，那么 `scheduler` 描述雷达的操作计划：什么时候看哪个方向。

这一步我们只学习 scan schedule：

- `config.scan` 保存 scan grid，比如 azimuths 和 elevations。
- `ScriptedScanScheduler` 把 scan grid 展开成一串 `DwellTask`。
- 每个 `DwellTask` 表示一次 dwell / CPI：雷达在一个时间段内看一个 look direction。

Mental model: 雷达不是一次看完整空间，而是按 dwell 一个方向一个方向地看。scheduler 负责把“我要扫这些方向”变成具体执行顺序。

In [15]:
from radar_basics.scheduler import ScriptedScanScheduler


scheduler = ScriptedScanScheduler(
    azimuths_deg=config.scan.azimuths_deg,
    elevations_deg=config.scan.elevations_deg,
    prf_hz=radar.waveform.prf_hz,
    num_pulses=radar.waveform.num_pulses,
    mode=config.scan.mode,
)

tasks = scheduler.tasks(num_scan_cycles=config.run.num_scan_cycles)

print("Config scan -> scheduler -> dwell tasks")
print(f"config.scan type:     {type(config.scan).__name__}")
print(f"scheduler type:       {type(scheduler).__name__}")
print(f"azimuths_deg:         {scheduler.azimuths_deg}")
print(f"elevations_deg:       {scheduler.elevations_deg}")
print(f"num_scan_cycles:      {config.run.num_scan_cycles}")
print(f"tasks count:          {len(tasks)}")

expected_tasks = (
    len(config.scan.azimuths_deg)
    * len(config.scan.elevations_deg)
    * config.run.num_scan_cycles
)
print(f"expected tasks count: {expected_tasks}")

print("\nFirst few DwellTask objects:")
for task in tasks[:6]:
    print(
        f"id={task.id:2d} "
        f"look=({task.look_az_deg:5.1f} deg az, {task.look_el_deg:5.1f} deg el) "
        f"start={task.start_time_s:.6f} s "
        f"duration={task.duration_s:.6f} s "
        f"center={task.center_time_s:.6f} s "
        f"pulses={task.num_pulses}"
    )

print("\nLast DwellTask:")
last_task = tasks[-1]
print(last_task)

print("\nScan order as (azimuth, elevation):")
print([(task.look_az_deg, task.look_el_deg) for task in tasks])

Config scan -> scheduler -> dwell tasks
config.scan type:     ScanConfig
scheduler type:       ScriptedScanScheduler
azimuths_deg:         (-10.0, 0.0, 10.0)
elevations_deg:       (-5.0, 0.0, 5.0)
num_scan_cycles:      2
tasks count:          18
expected tasks count: 18

First few DwellTask objects:
id= 0 look=(-10.0 deg az,  -5.0 deg el) start=0.000000 s duration=0.001600 s center=0.000800 s pulses=16
id= 1 look=(  0.0 deg az,  -5.0 deg el) start=0.001600 s duration=0.001600 s center=0.002400 s pulses=16
id= 2 look=( 10.0 deg az,  -5.0 deg el) start=0.003200 s duration=0.001600 s center=0.004000 s pulses=16
id= 3 look=(-10.0 deg az,   0.0 deg el) start=0.004800 s duration=0.001600 s center=0.005600 s pulses=16
id= 4 look=(  0.0 deg az,   0.0 deg el) start=0.006400 s duration=0.001600 s center=0.007200 s pulses=16
id= 5 look=( 10.0 deg az,   0.0 deg el) start=0.008000 s duration=0.001600 s center=0.008800 s pulses=16

Last DwellTask:
DwellTask(id=17, mode='search', look_az_deg=10.0, lo

### 观察这个输出

这里的关键对象是 `DwellTask`。它不是目标，也不是数据；它是一条执行指令：

```text
at this time, use this PRF and this many pulses, look at this az/el direction
```

几个重要字段：

- `look_az_deg` / `look_el_deg`：这次 dwell 的波束指向。
- `start_time_s`：这次 dwell 开始时间。
- `num_pulses` 和 `prf_hz`：这次 dwell 的 CPI 设置。
- `duration_s`：由 `num_pulses / prf_hz` 推导出来。
- `center_time_s`：这次 dwell 的中心时间，后面常用于 target truth snapshot 和 detection timestamp。

暂停点：请确认你能解释为什么 3 个 azimuth、3 个 elevation、2 个 scan cycles 会产生 18 个 `DwellTask`。

### My understanding after Step 3C

Scheduler 的作用是把 scan grid 展开成时间轴上的一串 `DwellTask`。

这里的 scan grid 可以理解为：按照离散的 azimuth/elevation 角度，把雷达要观察的空域划分成一组 look directions。

每个 `DwellTask` 表示一个 dwell / CPI 的执行指令：

```text
在某个时间段内，
雷达看向某个 az/el 方向，
用指定 PRF 发射指定数量的 pulses，
并收集这一段 CPI 的回波数据。
```

所以 scheduler 本身不关心目标在哪里，也不生成 IQ 数据。它只负责回答：

```text
雷达在什么时间，看哪个方向，看多久？
```

## Step 4: One Dwell Raw IQ

现在把前面三个 runtime objects 合在一起：

```text
radar + scene + one DwellTask -> RawDwellData
```

这一步调用 `synthesize_dwell()`，生成一个 dwell / CPI 的 raw complex IQ 数据。

Mental model: 一个目标在 raw IQ 里表现为三件事：

- fast-time delay：目标距离决定回波延迟。
- slow-time phase progression：径向速度决定 pulse-to-pulse phase rotation。
- array-space phase slope：目标角度决定阵元之间的 phase pattern。

这一步先只看数据结构和 shape，不急着做 signal processing。

In [ ]:
import numpy as np

from radar_basics import synthesize_dwell


task = tasks[0]
raw = synthesize_dwell(
    radar=radar,
    scene=scene,
    task=task,
    rng=np.random.default_rng(config.run.seed),
)

print("Inputs to synthesize_dwell")
print(f"radar type: {type(radar).__name__}")
print(f"scene type: {type(scene).__name__}")
print(f"task:       {task}")

print("\nRawDwellData")
print(f"raw type:      {type(raw).__name__}")
print(f"raw.iq dtype:  {raw.iq.dtype}")
print(f"raw.iq shape:  {raw.iq.shape}")
print(
    "expected:      "
    f"({radar.array.num_y}, {radar.array.num_x}, "
    f"{task.num_pulses}, {radar.waveform.num_fast_time_samples})"
)

print("\nRaw axes")
print(f"fast_time_s shape:        {raw.axes.fast_time_s.shape}")
print(f"pulse_times_s shape:      {raw.axes.pulse_times_s.shape}")
print(f"array_positions_m shape:  {raw.axes.array_positions_m.shape}")
print(f"first 5 fast-time samples: {raw.axes.fast_time_s[:5]}")
print(f"pulse times:               {raw.axes.pulse_times_s}")

print("\nTruth snapshot attached to this dwell")
truth = raw.truth[0]
print(f"truth type:           {type(truth).__name__}")
print(f"truth time approx:    task.center_time_s = {task.center_time_s:.6f} s")
print(f"range_m:              {truth.range_m:.2f}")
print(f"az_deg / el_deg:      {truth.az_deg:.2f} / {truth.el_deg:.2f}")
print(f"radial_velocity_mps:  {truth.radial_velocity_mps:.2f}")

magnitude = np.abs(raw.iq)
max_index = np.unravel_index(np.argmax(magnitude), magnitude.shape)
print("\nWhere is the largest IQ magnitude?")
print("index order: (array_y, array_x, pulse, fast_time)")
print(f"max index:     {max_index}")
print(f"max magnitude: {magnitude[max_index]:.3e}")

power_by_fast_time = np.max(magnitude, axis=(0, 1, 2))
active_fast_time_indices = np.flatnonzero(power_by_fast_time > 0.0)
print("\nFast-time samples containing nonzero echo energy")
print(f"first active index: {active_fast_time_indices[0]}")
print(f"last active index:  {active_fast_time_indices[-1]}")
print(f"active sample count: {len(active_fast_time_indices)}")

Inputs to synthesize_dwell
radar type: RadarSystem
scene type: Scene
task:       DwellTask(id=0, mode='search', look_az_deg=-10.0, look_el_deg=-5.0, start_time_s=0.0, prf_hz=10000.0, num_pulses=16)

RawDwellData
raw type:      RawDwellData
raw.iq dtype:  complex128
raw.iq shape:  (4, 4, 16, 1000)
expected:      (4, 4, 16, 1000)

Raw axes
fast_time_s shape:        (1000,)
pulse_times_s shape:      (16,)
array_positions_m shape:  (4, 4, 3)
first 5 fast-time samples: [0.e+00 1.e-07 2.e-07 3.e-07 4.e-07]
pulse times:               [0.     0.0001 0.0002 0.0003 0.0004 0.0005 0.0006 0.0007 0.0008 0.0009
 0.001  0.0011 0.0012 0.0013 0.0014 0.0015]

Truth snapshot attached to this dwell
truth type:           TargetSnapshot
truth time approx:    task.center_time_s = 0.000800 s
range_m:              1500.00
az_deg / el_deg:      0.00 / 0.00
radial_velocity_mps:  0.00

Where is the largest IQ magnitude?
index order: (array_y, array_x, pulse, fast_time)
max index:     (np.int64(0), np.int64(0), np.

### 观察这个输出

`raw.iq` 是这个 codebase 里第一个真正的 waveform-level 数据张量：

```text
raw.iq shape = (num_y, num_x, num_pulses, num_fast_time)
```

四个维度分别表示：

- `num_y`：阵列第一个空间维度。
- `num_x`：阵列第二个空间维度。
- `num_pulses`：slow time / pulse index。
- `num_fast_time`：fast time / ADC sample index。

这一步的核心是：`synthesize_dwell()` 把三类 runtime object 合并了起来。

```text
RadarSystem: 这台雷达怎么发、怎么收
Scene:       目标真实在哪里、怎么运动
DwellTask:   这一段时间雷达看哪个方向
```

暂停点：请确认你能解释为什么 raw IQ 需要同时有 array、pulse、fast-time 三类维度。

### Reference: `synthesize_dwell()` implements this signal model

对一个 target，`synthesize_dwell()` 实现的核心 raw IQ 公式可以写成：

$$
x_{e,p,n}
=
A_p
\cdot
e^{-j\frac{4\pi R_p}{\lambda}}
\cdot
e^{j\frac{2\pi}{\lambda}\mathbf{r}_e \cdot \mathbf{u}_p}
\cdot
s(t_n - \tau_p)
$$

这里 $e$ 是 array element index，$p$ 是 pulse index，$n$ 是 fast-time sample index。因此 $x_{e,p,n}$ 就是某个阵元、某个 pulse、某个 fast-time sample 上的 complex IQ value。

各项含义：

1. 双程延迟

   $$
   \tau_p = \frac{2R_p}{c}
   $$

   $R_p$ 是目标在第 $p$ 个 pulse 时刻的 range。这个 delay 决定目标回波落在 fast-time 轴上的哪个位置。

2. Delayed LFM chirp

   $$
   s(t) = e^{j\pi Kt^2}, \quad 0 \le t < T_p, \quad K = \frac{B}{T_p}
   $$

   $s(t_n - \tau_p)$ 表示目标回波是一个被 delay 之后的 LFM chirp。

3. 双程传播相位

   $$
   e^{-j\frac{4\pi R_p}{\lambda}}
   $$

   如果目标在运动，$R_p$ 会随着 pulse index $p$ 改变，于是这个相位会在 slow time 上旋转，形成 Doppler 信息。

4. 阵列空间相位

   $$
   e^{j\frac{2\pi}{\lambda}\mathbf{r}_e \cdot \mathbf{u}_p}
   $$

   $\mathbf{r}_e$ 是第 $e$ 个阵元的位置，$\mathbf{u}_p$ 是雷达到目标方向的 unit direction vector。不同阵元位置不同，所以相位不同；这个 spatial phase pattern 后面用于 angle beamforming。

5. 幅度项

   $$
   A_p
   =
   \sqrt{\frac{P_t N \lambda^2 \sigma}{(4\pi)^3 R_p^4 L}}
   \cdot
   G_{tx}(\mathbf{u}_{look}, \mathbf{u}_p)
   $$

   这里 $P_t$ 是 transmit peak power，$N$ 是阵元数量，$\lambda$ 是 wavelength，$\sigma$ 是 target RCS，$L$ 是 system loss。$G_{tx}$ 表示这次 dwell 的 beam look direction 和目标真实方向的匹配程度。

多个 target 的情况，就是把“一个 target 的公式”线性叠加：

$$
x_{e,p,n} = \sum_{q \in targets} x^{(q)}_{e,p,n} + w_{e,p,n}
$$

$w_{e,p,n}$ 是 complex Gaussian noise。

一句话总结：`synthesize_dwell()` implement 的就是这个公式。它对每个 target、每个 pulse、每个阵元，把 delayed LFM chirp、传播相位、阵列相位、transmit gain 和 radar-equation amplitude 写进 raw IQ tensor；多个 target 线性叠加，最后加噪声。

### 我的理解

到目前为止，这个 codebase 的主线是：

```text
YAML spec
-> Python dict
-> dataclass config
-> runtime objects: radar / scene / tasks
-> synthesize_dwell()
-> one CPI raw IQ data
```

YAML 文件可以理解为一次 radar simulation 的 specification。它用人类容易阅读和修改的方式，描述雷达系统、目标场景、扫描方式、处理参数和运行参数。

`load_config()` 做的事情，是把这份 YAML 先读成 Python dictionary，再转换成结构化的 dataclass config。这个阶段本质上还是同一份参数，只是组织方式变得更适合 Python 程序使用。

然后 codebase 会根据 config 构建几个重要的 runtime objects：

- `radar`：描述这台 2D phased-array radar 的硬件和 waveform model，比如阵列、LFM pulse、carrier frequency、PRF、range/velocity resolution 等。
- `scene`：描述真实世界里的目标状态和运动模型，比如目标位置、速度、RCS，以及某个时间点的 range/az/el/radial velocity。
- `tasks`：由 scheduler 生成，描述雷达在时间轴上的操作计划。每个 `DwellTask` 表示某个时间段内，雷达看向某个 az/el 方向，并收集一个 CPI 的数据。

所以可以把 `radar`、`scene`、`tasks` 理解成把仿真的“舞台”和“参与者”搭建好了：

```text
radar: 雷达怎么发、怎么收
scene: 真实世界里有什么目标
task:  这一段时间雷达看哪个方向
```

接下来，`synthesize_dwell(radar, scene, task)` 就让这三者发生 interaction。它根据雷达参数、目标真值和当前 dwell 的 look direction，生成这一段 CPI 内接收机看到的 waveform-level raw complex IQ data。

也就是说：

```text
synthesize_dwell()
= 把目标的 range / velocity / angle / RCS
  通过雷达信号模型
  写进 raw IQ tensor
```

输出的 `raw.iq` 就是一个 dwell 的接收数据：

```text
raw.iq shape = (array_y, array_x, pulse, fast_time)
```

- array 维度承载 angle / spatial phase 信息。
- pulse 维度承载 Doppler / velocity 信息。
- fast-time 维度承载 range / delay 信息。

一个重要补充是：`radar`、`scene` 不只是“存参数”。它们也有 runtime behavior，比如 `radar` 可以计算 wavelength、steering vector、noise power；`scene` 可以在任意时间生成 target snapshot。

## Step 5A: Range Compression

现在进入 processing chain 的第一步：range compression。

Step 4 里，目标回波在 raw IQ 的 fast-time 维度上还是一个 delayed LFM chirp。Range compression 的作用是用 matched filter 把这个 chirp 压成一个 sharp peak。

Mental model:

```text
raw delayed chirp along fast time
-> matched filtering with transmitted chirp
-> range peak
```

这一步只处理 fast-time 维度。array 维度和 pulse 维度还保留着，后面会继续用于 angle 和 Doppler processing。

In [ ]:
from radar_basics import SPEED_OF_LIGHT, range_compress


range_data = range_compress(raw, radar)

range_axis_m = (
    np.arange(range_data.shape[-1], dtype=float)
    * SPEED_OF_LIGHT
    / (2.0 * radar.waveform.sample_rate_hz)
)

print("Range compression")
print(f"raw.iq shape:      {raw.iq.shape}")
print(f"range_data shape:  {range_data.shape}")
print("shape order:       (array_y, array_x, pulse, range_bin)")

raw_mag = np.abs(raw.iq)
range_mag = np.abs(range_data)

raw_peak_index = np.unravel_index(np.argmax(raw_mag), raw_mag.shape)
range_peak_index = np.unravel_index(np.argmax(range_mag), range_mag.shape)

print("\nPeak before and after range compression")
print("index order: (array_y, array_x, pulse, fast_time/range_bin)")
print(f"raw peak index:        {raw_peak_index}")
print(f"compressed peak index: {range_peak_index}")

peak_range_bin = range_peak_index[-1]
print(f"compressed peak range bin: {peak_range_bin}")
print(f"compressed peak range:     {range_axis_m[peak_range_bin]:.2f} m")
print(f"truth range:               {raw.truth[0].range_m:.2f} m")
print(f"range bin spacing:         {range_axis_m[1] - range_axis_m[0]:.2f} m")

single_channel_raw = raw_mag[0, 0, 0]
single_channel_compressed = range_mag[0, 0, 0]

print("\nSingle channel width intuition")
print(f"raw active sample count:        {np.count_nonzero(single_channel_raw > 0.0)}")
print(f"compressed samples above 50% peak: {np.count_nonzero(single_channel_compressed > 0.5 * single_channel_compressed.max())}")

### 观察这个输出

`range_compress(raw, radar)` 的输入是 `RawDwellData`，输出还是一个 complex ndarray：

```text
range_data shape = (array_y, array_x, pulse, range_bin)
```

shape 看起来和 `raw.iq` 一样，但最后一个维度的含义变了：

- range compression 之前：最后一维是 raw fast-time samples，里面是一段 delayed chirp。
- range compression 之后：最后一维可以理解成 range bins，目标会变成比较尖的 peak。

这一步只回答一个问题：目标在哪个 range bin？

暂停点：请确认你能解释为什么 range compression 只改变 fast-time 维度的含义，而没有消除 array 维度和 pulse 维度。